# Titanic - Machine Learning From Disaster (Kaggle Challenge)
## Predictive model to ascertain likelihood of survival at the (unfortunate) Titanic based on given parameters. 
### From the perspective of a Year 1 - Business Analytics Student

### Would follow Ken Jee's approach to analyzing the data. <br>Link to his YouTube channel: (https://www.youtube.com/@KenJee_ds/)

## *__Data Overview__* (WIP)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression #For Logistic Regression

## *__Preliminary Data Loading and Investigation__*

### 1. Data Loading

In [ ]:
train_set = pd.read_csv("/Users/geolangsatnarzary/Study - NTU_NBS/01 - A-Career/Kaggle/train.csv")
test_set = pd.read_csv("/Users/geolangsatnarzary/Study - NTU_NBS/01 - A-Career/Kaggle/test.csv")

# Pre-defining the data sets with training or testing indicators for future data merges
train_set["train_test"] = 1
test_set["train_test"] = 0

#Setting all values of test-set to NaN
test_set["Survived"] = np.NaN

# Full merged data for future use
df = pd.concat([train_set,test_set])

### 2. Understanding the dataset and central tendencies of numeric data

In [ ]:
df.info()
print(df.head(10))

# Training Set Data: Statistical analysis on the numeric data along with central tendencies of data
train_set.describe()

### 3. Seperating numeric and categorical data for further analysis 

In [ ]:
# Separating Numeric and Categorical Data 
train_num = train_set[["Age","SibSp","Parch","Fare"]]
train_cat = train_set[["Survived","Pclass","Sex","Ticket","Cabin","Embarked"]]

### 4. Histograms to look at numeric data distribution 

In [ ]:
%matplotlib inline
for i in train_num.columns:
    plt.hist(train_num[i])
    plt.title(i)
    plt.show()

##### Note: Only "Age" variable is somewhat normally distributed but others are not. <br> Will decide to normalize based on further data investigation.

### 5. Correlation between numeric values

In [ ]:
print(train_num.corr())
sns.heatmap(train_num.corr())

##### "Parch" and "SibSp" have a correlation of 0.414838 as Parents, Children and Siblings (Families) tend to travel together. <br> This preliminary investigation ensures that we eliminate the risk of Multi-Collinearity during Regression.

### 6. Survival rate across numeric values

In [ ]:
pd.pivot_table(train_set, index='Survived', values=['Age','SibSp','Parch','Fare'])

##### Note: The result doesn't give us straight-forward insights but it is possible that: <br>
#####            1. Children have higher chance of surviving.
#####            2. People who paid more Fare have higher chance of surviving.

### 7. Barplots for Categorical Variables

In [ ]:
for i in train_cat.columns:
    sns.barplot(x= train_cat[i].value_counts().index,y = train_cat[i].value_counts()).set_title(i)
    plt.show()

##### Note: The above graphs give us an idea on how the data are distributed across different categorical data. <br> Particularly "Cabin" and "Ticket" could be further improved upon using feature engineering.

### 8. Survival rates across categorical variables

In [ ]:
print(pd.pivot_table(train_set, index="Survived", columns = "Pclass", values = "Ticket", aggfunc="count"),"\n")
print(pd.pivot_table(train_set, index="Survived", columns = "Sex", values = "Ticket", aggfunc="count"),"\n")
print(pd.pivot_table(train_set, index="Survived", columns = "Embarked", values = "Ticket", aggfunc="count"),"\n")

## *__Feature Engineering__*

## 1. Cabin - We simplify Cabin variable categories. <br> 
### Additionally, we see if purchase of multiple cabins has any impact. 

### 1.1 Splitting people with multiple cabins

In [ ]:
train_set['cabin_multiple'] = [0 if pd.isna(x) else len(x.split(' ')) for x in train_set['Cabin']]
train_set['cabin_multiple'].value_counts()

### 1.2 Survival rate for people with multiple cabins.

In [ ]:
print(pd.pivot_table(train_set, index="Survived", columns = "cabin_multiple", values = "Ticket", aggfunc="count"),"\n")

##### Note: Doesn't seem like that was the case necessarily.

### 1.3 Creating Categories based on first alphabet of the Cabins.
##### Based on the assumption that each class/area of cabin would be grouped together (as in Hotels).

In [ ]:
train_set['cabin_adv'] = train_set['Cabin'].astype(str).str[0]
print(train_set['cabin_adv'].value_counts())

### 1.4 Survival rate based on the new Cabin values

In [ ]:
print(pd.pivot_table(train_set, index="Survived", columns = "cabin_adv", values = "Ticket", aggfunc="count"),"\n")

### Conclusion:<br> This feature engineering provided us with clear insight that people with Letter cabin numbers had clearly higher chance of surviving than those with null values in the Cabin column.

## 2. Deep-Dive into the "Ticket" Values

### 2.1 Separate Numeric and Non-Numeric Tickets

In [ ]:
#If any type of Non-numeric value is in the ticket - it will be recorded in 'letter ticket' otherwise 'numeric_ticket' = 1
train_set['numeric_ticket'] = [1 if x.isnumeric() else 0 for x in train_set['Ticket']]
train_set['letter_ticket'] = [''.join(x.split(' ')[:-1]).replace('.','').replace('/','').lower() if len(x.split(' ')[:-1])>0 else 0 for x in train_set['Ticket']]

train_set['numeric_ticket'].value_counts()

In [ ]:
train_set['letter_ticket'].value_counts()

##### Note: Right now, the 'letter_ticket' values seem a little ambiguous as well. <br>We proceed to see if survival rate matters for if the ticket was numeric or not.

### 2.2 Survival rate based on Numeric or Letter Tickets

In [ ]:
pd.pivot_table(train_set, index="Survived", columns='numeric_ticket', values = "Ticket", aggfunc="count")

In [ ]:
pd.pivot_table(train_set, index="Survived", columns='letter_ticket', values = "Ticket", aggfunc="count")

### Conclusion: <br>The proportion of people survived seems to be similar to those of the aggregate, so seems like there isn't much significance if the Tickets had letters or numbers in it. 

## 3. Importance of Titles - "Mr", "Mrs", "Captain", "Dr", "Col".


### 3.1 Data Preparation

In [ ]:
train_set['name_title'] = [x.split(',')[1].split(".")[0].strip() for x in train_set['Name']]

In [ ]:
train_set['name_title'].value_counts()

### 3.2 Plotting the Survival Rate based on Titles

In [ ]:
pd.pivot_table(train_set, index="Survived", columns='name_title', values = "Ticket", aggfunc="count")

### Conclusion: <br>1. The Captain of the ship went down with the ship. (As depicted in the fictional movie created by James Cameroon) <br>2. Miss, Mrs and Master survived more in general than Mr, which could imply children and women were given preference. <br>3. Special titles such as Lady, Mlle (Mademoiselle), Mme (Madame), Sir and the Countess, also received preferencial treatment with 100% Survival rate. (6/6 Survived) 

# Model Building
## Data Preprocessing
### 1. Extending the same features engineering in training set to the complete dataset (train+test).

In [ ]:
df['cabin_multiple'] = [0 if pd.isna(x) else len(x.split(' ')) for x in df['Cabin']]
df['cabin_adv'] = [str(x)[0] for x in df['Cabin']]
df['numeric_ticket'] = [1 if x.isnumeric() else 0 for x in df['Ticket']]
df['ticket_letters'] = [''.join(x.split(' ')[:-1]).replace('.','').replace('/','').lower() if len(x.split(' ')[:-1]) > 0 else 0 for x in df['Ticket']]
df['name_title'] = [x.split(',')[1].split(".")[0].strip() for x in df['Name']]

### 2. Replacing null values in continuous data with mean(normally distributed) and median(not normally distributed) of the column.

In [ ]:
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

### 3. Drop null 'embarked' rows. Only 2 instances in training and 0 in test.

In [ ]:
df.dropna(subset=['Embarked'], inplace=True)

### 4. Normalized Fare price

In [ ]:
df['norm_fare'] = np.log(df.Fare+1)
df['norm_fare'].hist()

### 5. Converted PClass to Categorical variable

In [ ]:
df['Pclass'] = df['Pclass'].astype(str)

### 6. Created Dummy variables for categories

In [ ]:
df_dummies = pd.get_dummies(df[['Pclass','Sex','Age','SibSp','Parch','norm_fare','Embarked','cabin_adv','cabin_multiple','numeric_ticket','name_title','train_test']])

In [ ]:
X_train = df_dummies[df.train_test == 1].drop(['train_test'], axis=1)
X_test = df_dummies[df.train_test == 0].drop(['train_test'], axis=1)

In [ ]:
y_train = df[df.train_test == 1].Survived
y_train.shape

### 7. Scaled data

In [ ]:
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
df_dummies_scaled = df_dummies.copy()
print(len(df_dummies_scaled))
df[['Age','SibSp','Parch','norm_fare']]= scale.fit_transform(df_dummies_scaled[['Age','SibSp','Parch','norm_fare']])

X_train_scaled = df_dummies_scaled[df_dummies_scaled.train_test == 1].drop(['train_test'], axis =1)
print(len(X_train_scaled))
print(X_train_scaled)
X_test_scaled = df_dummies_scaled[df_dummies_scaled.train_test == 0].drop(['train_test'], axis =1)

y_train = df[df.train_test==1].Survived

## Model Performance Comparison with Baseline (Naive Bayes)

### We examine a few models and validate their scores in comparison to the Baseline for which we will use Naive Bayes.<br><br>Importing necessary packages. 

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn import tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

### 1. Gaussian Naive Bayes

In [ ]:
gnb = GaussianNB()
cv = cross_val_score(gnb, X_train_scaled, y_train, cv=5)
print(cv)
print(cv.mean())

### 2. Logistic Regression

In [ ]:
lr = LogisticRegression(max_iter=2000)
cv = cross_val_score(lr, X_train_scaled,y_train,cv=5)
print(cv)
print(cv.mean())

### 3. Decision Tree Classifier

In [ ]:
dt = tree.DecisionTreeClassifier(random_state=1)
cv = cross_val_score(dt,X_train_scaled, y_train, cv=5)
print(cv)
print(cv.mean())

### 4. K Neighbors Classifier

In [ ]:
knn = KNeighborsClassifier()
cv = cross_val_score(knn, X_train_scaled,y_train, cv=5)
print(cv)
print(cv.mean())

### 5. Random Forest Classifier

In [ ]:
rf = RandomForestClassifier(random_state=1)
cv = cross_val_score(rf, X_train_scaled, y_train, cv=5)
print(cv)
print(cv.mean())

### 6. C-Support Vector Classifier

In [ ]:
svc = SVC(probability=True)
cv = cross_val_score(svc,X_train_scaled, y_train, cv=5)
print(cv)
print(cv.mean())

### 7. Additionally, trying *_"XGBoost Classifier"_* which has generally performed well for other projects.

In [ ]:
from xgboost import XGBClassifier
xgb = XGBClassifier(random_state=1)
cv = cross_val_score(xgb, X_train_scaled, y_train, cv=5)
print(cv)
print(cv.mean())

### 8. Voting Classifier

In [ ]:
from sklearn.ensemble import VotingClassifier
voting_clf = VotingClassifier(estimators=[('lr',lr),('knn',knn),('rf',rf),('gnb',gnb),('svc',svc),('xgb',xgb)], voting='soft')
cv = cross_val_score(voting_clf, X_train_scaled, y_train, cv=5)
print(cv)
print(cv.mean())

# *__Preparing a Base Submission Based on Voting Classifier__*

In [ ]:
voting_clf.fit(X_train_scaled, y_train)
y_hat_base_vc = voting_clf.predict(X_test_scaled).astype(int)
basic_submission = {'PassengerId': test_set.PassengerId, 'Survived': y_hat_base_vc}
base_submission = pd.DataFrame(data=basic_submission)
base_submission.to_csv('base_submission.csv', index=False)